In [1]:
import os
import shutil
import numpy as np
from config import Config
from env import MySumoEnv

_NEEDED_WORKER_FILES = ("detectors.add.xml", "network.net.xml", "run.sumocfg")

def prepare_worker_dir(worker_idx=0):
    if hasattr(Config, "ensure_dirs"):
        Config.ensure_dirs()
    else:
        os.makedirs(Config.WORKERS_ROOT, exist_ok=True)
    src = Config.BASE_WORK_DIR
    dst = Config.worker_dir(worker_idx)
    os.makedirs(dst, exist_ok=True)
    for name in _NEEDED_WORKER_FILES:
        src_path = os.path.join(src, name)
        dst_path = os.path.join(dst, name)
        if not os.path.exists(src_path):
            raise FileNotFoundError(f"No utils file: {src_path}")
        shutil.copy2(src_path, dst_path)
    os.makedirs(os.path.join(dst, "dump"), exist_ok=True)
    return dst

def build_env(worker_idx=0, total_step=None, seed=42):
    rl_dir = prepare_worker_dir(worker_idx)
    answer_path = Config.answer_path_for(worker_idx)
    sumocfg_path = os.path.join(rl_dir, "run.sumocfg")

    if not os.path.exists(rl_dir):
        raise FileNotFoundError(f"No directory: {rl_dir}")
    if not os.path.exists(sumocfg_path):
        raise FileNotFoundError(f"No run.sumocfg: {sumocfg_path}")
    if not os.path.exists(answer_path):
        raise FileNotFoundError(f"No answer.csv: {answer_path}")

    if total_step is None:
        total_step = Config.TOTAL_STEP

    env = MySumoEnv(
        rl_dir=rl_dir,
        sumo_binary=Config.SUMO_BINARY,
        origin_list=Config.ORIGIN_LIST,
        destination_list=Config.DESTINATION_LIST,
        input_interval=Config.INPUT_INTERVAL,
        detector_interval=Config.DETECTOR_INTERVAL,
        num_OD=Config.NUM_OD,
        state_dim=1,
        answer_dir=answer_path,
        total_step=int(total_step),
    )

    obs, info = env.reset(seed=seed)
    return env, obs, info

In [2]:
import numpy as np
from bayes_opt import BayesianOptimization

def _vector_to_action_plan(x, num_od, total_step, block_size, demand_max):
    n_blocks = int(np.ceil(total_step / block_size))

    if demand_max == 1:
        x_int = (x >= 0.5).astype(np.int32)
    else:
        x_int = np.clip(np.rint(x), 0, demand_max).astype(np.int32)

    action_plan = np.zeros((total_step, num_od), dtype=np.int32)
    k = 0
    for b in range(n_blocks):
        s = b * block_size
        e = min((b + 1) * block_size, total_step)
        for od in range(num_od):
            action_plan[s:e, od] = x_int[k]
            k += 1
    return action_plan

def _evaluate_episode(build_env_fn, worker_idx, total_step, action_plan, seed):
    env, _, _ = build_env_fn(worker_idx=worker_idx, total_step=total_step, seed=seed)
    total_reward = 0.0
    try:
        for t in range(total_step):
            _, reward, terminated, truncated, _ = env.step(action_plan[t])
            total_reward += float(reward)
            if terminated or truncated:
                break
    finally:
        env.close()
    return total_reward

def bayes_optimize(
    build_env_fn,
    worker_idx,
    num_od,
    total_step,
    init_points=0,
    n_iter=10,
    seed=42,
    block_size=1,
    demand_max=1,
    verbose=2,
):
    if block_size < 1:
        raise ValueError("block_size must be >= 1")
    if demand_max < 1:
        raise ValueError("demand_max must be >= 1")

    n_blocks = int(np.ceil(total_step / block_size))
    dim = n_blocks * num_od
    key_order = [f"x{i}" for i in range(dim)]
    pbounds = {k: (0.0, float(demand_max)) for k in key_order}

    eval_count = {"n": 0}

    def objective(**kwargs):
        x = np.array([kwargs[k] for k in key_order], dtype=np.float32)
        action_plan = _vector_to_action_plan(
            x=x,
            num_od=num_od,
            total_step=total_step,
            block_size=block_size,
            demand_max=demand_max,
        )
        score = _evaluate_episode(
            build_env_fn=build_env_fn,
            worker_idx=worker_idx,
            total_step=total_step,
            action_plan=action_plan,
            seed=seed + eval_count["n"],
        )
        eval_count["n"] += 1
        return float(score)

    optimizer = BayesianOptimization(
        f=objective,
        pbounds=pbounds,
        random_state=seed,
        verbose=verbose,
        allow_duplicate_points=True,
    )
    optimizer.maximize(init_points=init_points, n_iter=n_iter)

    best_x = np.array([optimizer.max["params"][k] for k in key_order], dtype=np.float32)
    best_actions = _vector_to_action_plan(
        x=best_x,
        num_od=num_od,
        total_step=total_step,
        block_size=block_size,
        demand_max=demand_max,
    )
    best_value = float(optimizer.max["target"])
    return best_actions, best_value, optimizer

In [3]:
import os
import sys
import json
import time
import tempfile
import subprocess
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
from config import Config
from trial_timing import reset_trial_times, append_trial_time

TRIALS = list(range(5))
MAX_WORKERS = len(TRIALS)
_trial_result_dir = RESULT_DIR if "RESULT_DIR" in globals() else Config.RESULT_DIR
os.makedirs(Config.RESULT_DIR, exist_ok=True)
reset_trial_times(_trial_result_dir)

_runner_source = "import os\nimport sys\nimport json\nimport time\nfrom pathlib import Path\n_scripts_dir = Path(os.environ['DODE_SCRIPTS_DIR']).resolve()\nos.chdir(_scripts_dir)\nif str(_scripts_dir) not in sys.path:\n    sys.path.insert(0, str(_scripts_dir))\ntrial = int(sys.argv[1])\n\nimport os\nimport shutil\nimport numpy as np\nfrom config import Config\nfrom env import MySumoEnv\n\n_NEEDED_WORKER_FILES = (\"detectors.add.xml\", \"network.net.xml\", \"run.sumocfg\")\n\ndef prepare_worker_dir(worker_idx=0):\n    if hasattr(Config, \"ensure_dirs\"):\n        Config.ensure_dirs()\n    else:\n        os.makedirs(Config.WORKERS_ROOT, exist_ok=True)\n    src = Config.BASE_WORK_DIR\n    dst = Config.worker_dir(worker_idx)\n    os.makedirs(dst, exist_ok=True)\n    for name in _NEEDED_WORKER_FILES:\n        src_path = os.path.join(src, name)\n        dst_path = os.path.join(dst, name)\n        if not os.path.exists(src_path):\n            raise FileNotFoundError(f\"No utils file: {src_path}\")\n        shutil.copy2(src_path, dst_path)\n    os.makedirs(os.path.join(dst, \"dump\"), exist_ok=True)\n    return dst\n\ndef build_env(worker_idx=0, total_step=None, seed=42):\n    rl_dir = prepare_worker_dir(worker_idx)\n    answer_path = Config.answer_path_for(worker_idx)\n    sumocfg_path = os.path.join(rl_dir, \"run.sumocfg\")\n\n    if not os.path.exists(rl_dir):\n        raise FileNotFoundError(f\"No directory: {rl_dir}\")\n    if not os.path.exists(sumocfg_path):\n        raise FileNotFoundError(f\"No run.sumocfg: {sumocfg_path}\")\n    if not os.path.exists(answer_path):\n        raise FileNotFoundError(f\"No answer.csv: {answer_path}\")\n\n    if total_step is None:\n        total_step = Config.TOTAL_STEP\n\n    env = MySumoEnv(\n        rl_dir=rl_dir,\n        sumo_binary=Config.SUMO_BINARY,\n        origin_list=Config.ORIGIN_LIST,\n        destination_list=Config.DESTINATION_LIST,\n        input_interval=Config.INPUT_INTERVAL,\n        detector_interval=Config.DETECTOR_INTERVAL,\n        num_OD=Config.NUM_OD,\n        state_dim=1,\n        answer_dir=answer_path,\n        total_step=int(total_step),\n    )\n\n    obs, info = env.reset(seed=seed)\n    return env, obs, info\n\nimport numpy as np\nfrom bayes_opt import BayesianOptimization\n\ndef _vector_to_action_plan(x, num_od, total_step, block_size, demand_max):\n    n_blocks = int(np.ceil(total_step / block_size))\n\n    if demand_max == 1:\n        x_int = (x >= 0.5).astype(np.int32)\n    else:\n        x_int = np.clip(np.rint(x), 0, demand_max).astype(np.int32)\n\n    action_plan = np.zeros((total_step, num_od), dtype=np.int32)\n    k = 0\n    for b in range(n_blocks):\n        s = b * block_size\n        e = min((b + 1) * block_size, total_step)\n        for od in range(num_od):\n            action_plan[s:e, od] = x_int[k]\n            k += 1\n    return action_plan\n\ndef _evaluate_episode(build_env_fn, worker_idx, total_step, action_plan, seed):\n    env, _, _ = build_env_fn(worker_idx=worker_idx, total_step=total_step, seed=seed)\n    total_reward = 0.0\n    try:\n        for t in range(total_step):\n            _, reward, terminated, truncated, _ = env.step(action_plan[t])\n            total_reward += float(reward)\n            if terminated or truncated:\n                break\n    finally:\n        env.close()\n    return total_reward\n\ndef bayes_optimize(\n    build_env_fn,\n    worker_idx,\n    num_od,\n    total_step,\n    init_points=0,\n    n_iter=10,\n    seed=42,\n    block_size=1,\n    demand_max=1,\n    verbose=2,\n):\n    if block_size < 1:\n        raise ValueError(\"block_size must be >= 1\")\n    if demand_max < 1:\n        raise ValueError(\"demand_max must be >= 1\")\n\n    n_blocks = int(np.ceil(total_step / block_size))\n    dim = n_blocks * num_od\n    key_order = [f\"x{i}\" for i in range(dim)]\n    pbounds = {k: (0.0, float(demand_max)) for k in key_order}\n\n    eval_count = {\"n\": 0}\n\n    def objective(**kwargs):\n        x = np.array([kwargs[k] for k in key_order], dtype=np.float32)\n        action_plan = _vector_to_action_plan(\n            x=x,\n            num_od=num_od,\n            total_step=total_step,\n            block_size=block_size,\n            demand_max=demand_max,\n        )\n        score = _evaluate_episode(\n            build_env_fn=build_env_fn,\n            worker_idx=worker_idx,\n            total_step=total_step,\n            action_plan=action_plan,\n            seed=seed + eval_count[\"n\"],\n        )\n        eval_count[\"n\"] += 1\n        return float(score)\n\n    optimizer = BayesianOptimization(\n        f=objective,\n        pbounds=pbounds,\n        random_state=seed,\n        verbose=verbose,\n        allow_duplicate_points=True,\n    )\n    optimizer.maximize(init_points=init_points, n_iter=n_iter)\n\n    best_x = np.array([optimizer.max[\"params\"][k] for k in key_order], dtype=np.float32)\n    best_actions = _vector_to_action_plan(\n        x=best_x,\n        num_od=num_od,\n        total_step=total_step,\n        block_size=block_size,\n        demand_max=demand_max,\n    )\n    best_value = float(optimizer.max[\"target\"])\n    return best_actions, best_value, optimizer\n\nimport time\nfrom trial_timing import reset_trial_times, append_trial_time\nimport json\nfrom config import Config\n\n_trial_result_dir = RESULT_DIR if \"RESULT_DIR\" in globals() else Config.RESULT_DIR\n\n_trial_t0 = time.perf_counter()\n\nblock_size = 1\ntrial_idx = trial\nseed_opt = int(100 + (trial_idx + 1))\n# Estimate seed (101-105) is kept separate from reproduce seeds (0-4).\nseed_eval = trial\n\nbest_actions, best_score, bo = bayes_optimize(\n    build_env_fn=build_env,\n    worker_idx=trial,\n    num_od=Config.NUM_OD,\n    total_step=Config.TOTAL_STEP,\n    init_points=0,\n    n_iter=300,\n    seed=seed_opt,\n    block_size=block_size,\n    demand_max=1,\n    verbose=2,\n)\n\nprint(\"Best total_reward:\", best_score)\nprint(\"Best action shape:\", best_actions.shape)\n\nenv, obs, info = build_env(worker_idx=trial, seed=seed_eval)\nprint(\"reset complete, obs shape:\", obs.shape)\n\nfor t in range(Config.TOTAL_STEP):\n    print(f\"step: {t+1}\")\n    action = best_actions[t]\n    obs, reward, terminated, truncated, info = env.step(action)\n\nenv.close()\n\ntrajectory = info.get(\"trajectory\", [])\nwith open(Config.RESULT_DIR + f\"/trajectory_{trial}.json\", \"w\", encoding=\"utf-8\") as f:\n    json.dump(trajectory, f, ensure_ascii=False, indent=2)\n\n_elapsed_path = os.environ.get('DODE_TRIAL_ELAPSED_PATH')\nif _elapsed_path:\n    with open(_elapsed_path, 'w', encoding='utf-8') as f:\n        json.dump({'trial': int(trial), 'elapsed_sec': float(time.perf_counter() - _trial_t0)}, f)\n"
_scripts_dir = Path(Config.RL_ROOT).resolve() / "scripts"

def _run_trial_process(trial, runner_path, tmp_dir):
    elapsed_path = Path(tmp_dir) / f"elapsed_{trial}.json"
    env = os.environ.copy()
    env["DODE_SCRIPTS_DIR"] = str(_scripts_dir)
    env["DODE_TRIAL_ELAPSED_PATH"] = str(elapsed_path)
    env["PYTHONUNBUFFERED"] = "1"
    cmd = [sys.executable, str(runner_path), str(trial)]
    started = time.perf_counter()
    print(f"[trial {trial}] started")
    proc = subprocess.run(cmd, cwd=str(_scripts_dir), env=env)
    elapsed = time.perf_counter() - started
    if proc.returncode != 0:
        raise RuntimeError(f"trial {trial} failed with exit code {proc.returncode}")
    if elapsed_path.exists():
        with open(elapsed_path, "r", encoding="utf-8") as f:
            payload = json.load(f)
        elapsed = float(payload.get("elapsed_sec", elapsed))
    return int(trial), float(elapsed)

with tempfile.TemporaryDirectory(prefix="dode_main_parallel_") as _tmp_dir:
    _runner_path = Path(_tmp_dir) / "trial_runner.py"
    _runner_path.write_text(_runner_source, encoding="utf-8")

    _results = []
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = {executor.submit(_run_trial_process, trial, _runner_path, _tmp_dir): trial for trial in TRIALS}
        for future in as_completed(futures):
            trial, elapsed = future.result()
            print(f"[trial {trial}] completed in {elapsed:.3f} sec")
            _results.append((trial, elapsed))

for trial, elapsed in sorted(_results):
    append_trial_time(_trial_result_dir, trial, elapsed)

[trial 0] started
[trial 1] started
[trial 2] started
[trial 3] started
[trial 4] started
[trial 3] completed in 4461.807 sec
[trial 4] completed in 4530.082 sec
[trial 1] completed in 4551.404 sec
[trial 0] completed in 6407.427 sec
[trial 2] completed in 8222.628 sec


In [4]:
import os
import shutil
from config import Config


def remove_workers_root():
                                                            
    root = os.path.abspath(Config.WORKERS_ROOT)
    project_root = os.path.abspath(Config.RL_ROOT)
    if os.path.basename(root) != "workers":
        raise RuntimeError(f"Refusing to delete non-workers path: {root}")
    if os.path.commonpath([root, project_root]) != project_root:
        raise RuntimeError(f"Refusing to delete path outside experiment root: {root}")
    if os.path.isdir(root):
        shutil.rmtree(root)


remove_workers_root()
